# AICE Associate 변주 문제 (기초+비지도학습 포함) - 혼잡함, 비혼잡함 예측 (분류)

### 이 노트북의 구성요소

아래 다이어그램은 이 노트북 해설에서 실제로 사용된 함수·클래스·모듈을 구간별로 모은 것입니다 (굵게 표시된 이름은 이 노트북에서 특별히 선택된 항목입니다).

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>

<div style="display:flex; flex-direction:column; gap:6px; width:fit-content;">
<pre class="mermaid">
flowchart LR
    subgraph LIB["라이브러리"]
        LIB1["<b>라이브러리</b><br/>numpy<br/>pandas<br/>pyplot<br/>seaborn"]
    end
    subgraph EXP["데이터 탐색"]
        EXP1["<b>데이터 탐색</b><br/>concat<br/>corr<br/>describe<br/>groupby<br/>head<br/>info"]
    end
    subgraph PREP["전처리"]
        PREP1["<b>전처리</b><br/><b>StandardScaler</b><br/>LabelEncoder<br/>cut<br/>describe<br/>drop<br/>drop_duplicates"]
    end
    LIB --> EXP --> PREP
    classDef libCls fill:#F0EEE8,stroke:#8A8478,color:#3A362E
    class LIB libCls
    classDef expCls fill:#E9F1F6,stroke:#5B7C99,color:#1E3548
    class EXP expCls
    classDef prepCls fill:#FAF3E7,stroke:#B08D57,color:#5C4720
    class PREP prepCls
</pre>
<pre class="mermaid">
flowchart LR
    subgraph ML["머신러닝"]
        ML1["<b>머신러닝</b><br/><b>ExtraTreesClassifier</b><br/><b>RandomizedSearchCV</b><br/><b>roc_auc_score</b><br/>DataFrame<br/>DecisionTreeClassifier<br/>classification_report"]
    end
    subgraph UNSUP["비지도학습"]
        UNSUP1["<b>비지도학습</b><br/><b>DBSCAN</b><br/><b>PCA</b><br/>Series<br/>value_counts"]
    end
    subgraph DL["딥러닝"]
        DL1["<b>딥러닝</b><br/><b>BatchNormalization</b><br/><b>ModelCheckpoint</b><br/><b>load_model</b><br/>Dense<br/>Dropout<br/>EarlyStopping"]
    end
    ML --> UNSUP --> DL
    classDef mlCls fill:#FBEEEA,stroke:#B5654A,color:#5C2A1B
    class ML mlCls
    classDef unsupCls fill:#F3EEF4,stroke:#8B6F8E,color:#402F42
    class UNSUP unsupCls
    classDef dlCls fill:#EBEEF6,stroke:#445577,color:#232C3D
    class DL dlCls
</pre>
</div>

<script>
mermaid.initialize({ startOnLoad: false, securityLevel: "loose" });
mermaid.run();
</script>
"""))

### 스톱워치

아래 셀을 실행하면 경과 시간이 표시됩니다. 문제를 풀기 시작하기 전에 먼저 실행하세요.

In [ ]:
from IPython.display import HTML, display

def show_stopwatch():
    display(HTML('''
<div id="stopwatch" style="font-size: 18px; font-weight: bold; color: #d93025; padding: 8px; border: 1px solid #d93025; border-radius: 5px; display: inline-block;">
    ⏱️ 경과 시간: <span id="time_str">00:00</span>
</div>
<script>
    var sec = 0;
    var timer = setInterval(function(){
        sec++;
        var m = Math.floor(sec / 60);
        var s = sec % 60;
        document.getElementById("time_str").innerText =
            (m < 10 ? "0" + m : m) + ":" + (s < 10 ? "0" + s : s);
    }, 1000);
</script>
'''))
show_stopwatch()

### 시나리오

국립공원 관리 기관은 주요 관광지(예: 설악산, 제주도 특정 해변)의 실시간 혼잡도를 예측하여 방문객 분산 및 시설 관리를 최적화하고자 합니다. 이를 통해 특정 시점에 과부하가 걸리는 지역에 선제적으로 인력을 배치하고, 방문객들에게 효율적인 동선을 안내함으로써 관광 경험의 질을 향상시키는 것을 목표로 합니다.

---

**[유의사항]**
- 답안은 각 문항 아래 표시된 `# (N) ...` 칸에 작성하세요.
- **정답/해설은 이 노트북 가장 아래 `## 해설` 섹션에 모아뒀습니다.** 먼저 스스로 풀어본 뒤에 확인하세요.
- 문제를 다 풀면 `## 자동 채점` 셀을 실행해서 점수를 확인할 수 있습니다 (해설을 보기 전에 먼저 채점해보세요).
- 이 노트북은 오리지널 창작 문제이며, 실제 AICE 샘플문항 원문을 복제하지 않습니다.
- 데이터 로더: `read_json` / 기본모델: `DecisionTreeClassifier` / 비교모델: `ExtraTreesClassifier` / 스케일러: `StandardScaler`

**[데이터 컬럼 설명]**

- congestion_level : 혼잡함, 비혼잡함
- daily_visitor_count : 일별 총 방문객 수
- location_type : 지역 유형(국립공원,해변,역사지구,도시)
- zone_code : 세부 구역 코드(A1, B2, C3 등 5~8개 코드값),혼잡도 가중치 (dim.csv 와 병합 키)
- day_of_week : 피처 컬럼
- seasonality_index : 피처 컬럼
- average_hourly_traffic : 피처 컬럼
- weather_condition : 피처 컬럼
- location_id : 식별자(모델링에 불필요)
- event_date : 이벤트 발생 날짜 (문자열, 전처리 단계에서 datetime 변환 후 파생 컬럼 추출용)
- (병합 후) dim_value : 구역별 평균 혼잡도 가중치

## 0. 데이터 준비

다음 문항을 풀기 전에 아래 코드를 실행하세요 (문제 데이터 2개 테이블을 생성합니다).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def synthesize_tables(seed=20006, n_rows=600):
    rng = np.random.default_rng(seed)
    n = n_rows

    base = rng.normal(loc=50, scale=15, size=n)
    outlier_idx = rng.choice(n, size=max(3, n // 50), replace=False)
    base[outlier_idx] += rng.choice([1, -1], size=len(outlier_idx)) * rng.uniform(80, 150, size=len(outlier_idx))

    main_df = pd.DataFrame({'daily_visitor_count': base.round(2)})

    cats = [f"Cat{i+1}" for i in range(rng.integers(3, 5))]
    main_df['location_type'] = rng.choice(cats, size=n)

    n_regions = rng.integers(5, 9)
    regions = [f"R{i+1:02d}" for i in range(n_regions)]
    main_df['zone_code'] = rng.choice(regions, size=n)

    for col in ['day_of_week', 'seasonality_index', 'average_hourly_traffic', 'weather_condition']:
        main_df[col] = rng.normal(0, 1, size=n).round(2)

    main_df['location_id'] = [f"ID{i:05d}" for i in range(n)]

    z = (base - base.mean()) / (base.std() + 1e-9)
    classes = ['혼잡함', '비혼잡함']
    if '분류' == "분류":
        if False and len(classes) >= 3:
            bins = np.quantile(z, [1 / 3, 2 / 3])
            idx = np.digitize(z, bins)
            main_df['congestion_level'] = [classes[i] for i in idx]
        else:
            prob = 1 / (1 + np.exp(-z))
            labels = (rng.random(n) < prob).astype(int)
            main_df['congestion_level'] = np.where(labels == 1, classes[0], classes[-1])
    else:
        noise = rng.normal(0, 5, size=n)
        main_df['congestion_level'] = (base * 1.5 + noise).round(2)

    for col in ['daily_visitor_count'] + ['day_of_week', 'seasonality_index', 'average_hourly_traffic', 'weather_condition'][:1]:
        na_idx = rng.choice(n, size=int(n * 0.03), replace=False)
        main_df.loc[na_idx, col] = np.nan

    event_dates = pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 730, size=n), unit="D")
    main_df["event_date"] = event_dates.strftime("%Y-%m-%d")

    dup_idx = rng.choice(n, size=max(2, int(n * 0.01)), replace=False)
    main_df = pd.concat([main_df, main_df.loc[dup_idx]], ignore_index=True)

    main_df = main_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    dim_df = pd.DataFrame({
        'zone_code': regions,
        "dim_value": rng.uniform(0.5, 2.0, size=n_regions).round(3),
    })
    return main_df, dim_df


main_df, dim_df = synthesize_tables()
main_df.to_json("data.json", index=False)
dim_df.to_csv("dim.csv", index=False)
print("데이터 저장 완료 - data.json:", main_df.shape, "/ dim.csv:", dim_df.shape)
main_df.head(4)

## <데이터 분석>

### 1. 라이브러리 임포트

pandas, numpy, matplotlib.pyplot, seaborn 을 각각 pd, np, plt, sns 별칭으로 임포트하세요.

In [ ]:
# (1) 여기에 답안코드를 작성하고 실행하세요



### 2. 데이터 로드 (read_csv/read_json)

`data.json` 를 `pd.read_json` 로 읽어 **my_data** 에, `dim.csv` 를 `pd.read_csv` 로 읽어 **dim_data** 에 각각 할당하세요.

In [ ]:
# (2) 여기에 답안코드를 작성하고 실행하세요



### 3. 데이터 구조 확인

`my_data.info()` 로 구조를 확인하고, `my_data.shape` 를 이용해 행 개수는 **n_rows**, 열 개수는 **n_cols** 변수에 저장하세요.

In [ ]:
# (3) 여기에 답안코드를 작성하고 실행하세요



### 4. 결측치 확인

my_data 의 `daily_visitor_count` 컬럼에 결측치(NaN)가 몇 개 있는지 `isna().sum()` 으로 확인하세요. 몇 개입니까?

In [ ]:
# (4) 정답을 answer_4 변수에 저장하세요 (실행하세요)

answer_4 = ""


### 5. 데이터 병합 (pd.merge)

`zone_code` 를 키로 my_data 와 dim_data 를 **left join** 하여 **data_merged** 에 저장하세요.

In [ ]:
# (5) 여기에 답안코드를 작성하고 실행하세요



### 6. 데이터 집계 (groupby)

`location_type` 별 `daily_visitor_count` 의 평균을 구해 **df_grp** 에 저장하세요.

In [ ]:
# (6) 여기에 답안코드를 작성하고 실행하세요



### 7. 조건부 인덱싱 (loc/iloc)

`loc` 를 사용해 `daily_visitor_count` 값이 상위 25%(0.75 분위수) 이상인 행만 선택해 **top_quartile_df** 에 저장하고, `iloc` 으로 그 중 처음 5행만 선택해 **top5_df** 에 저장하세요.

In [ ]:
# (7) 여기에 답안코드를 작성하고 실행하세요



### 8. 정렬 (sort_values)

data_merged 를 `daily_visitor_count` 기준 내림차순으로 정렬해서 **sorted_df** 에 저장하세요.

In [ ]:
# (8) 여기에 답안코드를 작성하고 실행하세요



### 9. 컬럼명 변경 (rename)

data_merged 의 `dim_value` 컬럼명을 `region_weight` 로 바꾼 사본을 **renamed_df** 에 저장하세요 (data_merged 자체는 바꾸지 않습니다).

In [ ]:
# (9) 여기에 답안코드를 작성하고 실행하세요



### 10. 데이터 합치기 (concat)

data_merged 를 위/아래 절반으로 나눈 뒤 `pd.concat` 으로 다시 이어붙여 **concat_df** 에 저장하세요 (행 방향, axis=0).

In [ ]:
# (10) 여기에 답안코드를 작성하고 실행하세요



### 11. 피벗/교차분석

`pivot_table` 로 `location_type` 별 `daily_visitor_count` 평균을 **pivot_result** 에 저장하세요.

In [ ]:
# (11) 여기에 답안코드를 작성하고 실행하세요



### 12. 시각화 (subplots)

지역 유형(국립공원,해변,역사지구,도시)(location_type) 분포의 countplot 과, 혼잡함, 비혼잡함 별 일별 총 방문객 수(daily_visitor_count) histplot 을 나란히 그리는 코드입니다.
빈칸 **(A)** 에 들어갈, 여러 그래프를 한 번에 그릴 때 쓰는 matplotlib 함수 이름은?

```python
fig, axes = plt.(A)(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='location_type', ax=axes[0])
sns.histplot(data=data_merged, x='daily_visitor_count', hue='congestion_level', ax=axes[1])
plt.show()
```

In [ ]:
# (12) 정답을 answer_12 변수에 저장하세요 (실행하세요)

answer_12 = ""


### 13. 시각화 2

혼잡함, 비혼잡함 별 일별 총 방문객 수(daily_visitor_count) 분포를 seaborn boxplot 으로 시각화하세요.

In [ ]:
# (13) 여기에 답안코드를 작성하고 실행하세요



### 14. 상관관계 히트맵

수치형 컬럼들 간의 상관관계를 `.corr()` 로 구해 **corr_df** 에 저장하고, seaborn heatmap 으로 시각화하세요.

In [ ]:
# (14) 여기에 답안코드를 작성하고 실행하세요



## <데이터 전처리>

### 15. 이상치 처리

IQR 기준(K=1.5)을 벗어나는 이상치 행을 제거하고, 식별자 컬럼도 삭제해서 **data_temp** 에 저장하는 코드입니다. 빈칸 **(A)** 에 들어갈, 행을 삭제할 때 쓰는 DataFrame 메서드 이름을 파악한 뒤, 아래 코드 전체를 (A)를 채워서 작성하고 실행하세요 (이후 문항들이 data_temp 를 사용합니다).

```python
q1 = data_merged['daily_visitor_count'].quantile(0.25)
q3 = data_merged['daily_visitor_count'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
data_temp = data_merged.(A)(data_merged[(data_merged['daily_visitor_count'] > upper_fence) | (data_merged['daily_visitor_count'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['location_id'])
data_temp = data_temp.reset_index(drop=True)
```

In [ ]:
# (15) 위 코드에서 (A)를 채운 전체 코드를 작성하고 실행하세요



### 16. 중복 데이터 제거

data_temp 에 완전히 동일한(중복) 행이 있는지 `duplicated()` 로 개수를 확인해 **dup_count** 에 저장하고, `drop_duplicates()` 로 중복 행을 제거한 뒤 인덱스를 리셋해서 다시 **data_temp** 에 저장하세요.

In [ ]:
# (16) 여기에 답안코드를 작성하고 실행하세요



### 17. 결측치 처리

다음은 data_temp 의 결측치를 대표값으로 채우는 코드인데, 실행하면 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fill_value = data_temp['daily_visitor_count'].mode()[0]
data_na = data_temp.fillna({'daily_visitor_count': fill_value})
data_na['day_of_week'] = data_na['day_of_week'].fillna(data_na['day_of_week'].mode()[0])
# (이 버전에는 의도한 것과 다른 결과를 내는 부분이 있습니다)
```

In [ ]:
# (17) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 18. 날짜 특성 추출

`event_date` 컬럼을 `pd.to_datetime()` 으로 datetime 타입으로 변환한 뒤, 요일(**day_of_week**, 0=월~6=일)과 주말 여부(**is_weekend**, 0/1) 파생 컬럼을 만들고, 원본 `event_date` 컬럼은 삭제해서 다시 data_na 에 저장하세요.

In [ ]:
# (18) 여기에 답안코드를 작성하고 실행하세요



### 19. 파생 변수 생성 (비율)

`daily_visitor_count` 값을 `day_of_week` 의 절댓값에 1을 더한 값으로 나눈 파생 변수를 **ratio_feature** 라는 새 컬럼으로 만들어 data_na 에 추가하세요.

In [ ]:
# (19) 여기에 답안코드를 작성하고 실행하세요



### 20. 구간화 (Binning)

`daily_visitor_count` 값을 `pd.cut()` 으로 3개 구간(Low/Medium/High)으로 나누고, 구간별 개수를 **bin_counts** 에 저장하세요 (data_na 자체는 변경하지 않아도 됩니다).

In [ ]:
# (20) 여기에 답안코드를 작성하고 실행하세요



### 21. 인코딩

`location_type` 는 원-핫 인코딩(get_dummies, drop_first=True), `zone_code` 는 sklearn LabelEncoder 로 인코딩해서 data_preset 에 저장하세요. 타깃 컬럼 `congestion_level` 도 LabelEncoder 로 정수 인코딩하세요.

In [ ]:
# (21) 여기에 답안코드를 작성하고 실행하세요



### 22. 데이터 분리

congestion_level 을 y, 나머지를 X 로 삼아 train_test_split 으로 분리하세요.
- test_size=0.3, random_state=7, stratify 옵션을 적용하세요
- 변수명: X_train, X_valid, y_train, y_valid

In [ ]:
# (22) 여기에 답안코드를 작성하고 실행하세요



### 23. 스케일링

StandardScaler 로 훈련/검증 데이터를 스케일링하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.fit_transform(X_valid)
```

In [ ]:
# (23) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 24. 스케일러 특성

StandardScaler 를 훈련 데이터에 적용하면, 결과 값의 분포는 이론적으로 어떤 특성을 가지게 됩니까?

In [ ]:
# (24) 정답을 answer_24 변수에 저장하세요 (실행하세요)

answer_24 = ""


## <AI 모델링>

### 25. 머신러닝 기본 (fit-predict)

DecisionTreeClassifier 로 모델을 하나 만들어 학습시키고, 검증데이터에 대한 예측값을 **pred_y** 에, `model.score(X_valid_scaled, y_valid)` 결과를 **model_score** 에 저장하세요. (변수명: model, pred_y, model_score)

In [ ]:
# (25) 여기에 답안코드를 작성하고 실행하세요



### 26. GridSearch 모델링

DecisionTreeClassifier 와 ExtraTreesClassifier 를 RandomizedSearchCV(무작위 5회 탐색, n_iter=5)(cv=10, scoring='f1_macro')로 탐색하고 학습하세요.
- max_depth 후보: [3, 5, 7]
- ExtraTreesClassifier 의 n_estimators 후보: [50, 100, 200]
- 변수명: gs_a (베이스 모델), gs_b (비교 모델)

In [ ]:
# (26) 여기에 답안코드를 작성하고 실행하세요



### 27. GridSearch 결과 확인

위 GridSearch에서 ExtraTreesClassifier 의 n_estimators 후보는 [50, 100, 200] 였습니다. GridSearchCV가 고를 수 있는 값의 후보 중 '가장 큰 값'은 얼마인가요?

In [ ]:
# (27) 정답을 answer_27 변수에 저장하세요 (실행하세요)

answer_27 = ""


### 28. 변수중요도

ExtraTreesClassifier 의 변수중요도 Top 15개(정렬 ascending=True)를 뽑아 bar 차트로 시각화하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_ / 100})
fi = fi.sort_values('importance', ascending=True)[:15]
sns.barplot(x='feature', y='importance', data=fi)
plt.show()
```

In [ ]:
# (28) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 29. 성능평가

검증데이터로 gs_a, gs_b 두 모델의 **roc_auc_score** 를 각각 계산해서 a_score, b_score 에 저장하세요 (predict_proba 의 양성 클래스(1) 확률을 사용하세요. 타깃은 인코딩 단계에서 이미 0/1 정수로 변환되어 있습니다).

In [ ]:
# (29) 여기에 답안코드를 작성하고 실행하세요



### 30. 모델 성능 비교

바로 위에서 계산한 a_score 와 b_score 를 비교했을 때, 어느 모델이 더 우수하다고 판단할 수 있습니까? (둘 중 하나를 실행 결과에 따라 답하세요)

In [ ]:
# (30) 정답을 answer_30 변수에 저장하세요 (실행하세요)

answer_30 = ""


### 31. 혼동행렬 및 분류 리포트

gs_b(ExtraTreesClassifier)의 검증데이터 예측값으로 confusion_matrix 를 구해 **cm** 에 저장하고 heatmap 으로 시각화한 뒤, classification_report 를 출력하세요.

In [ ]:
# (31) 여기에 답안코드를 작성하고 실행하세요



## <비지도학습>

### 32. 차원축소 (PCA)

PCA 로 X_train_scaled 를 2차원으로 축소해 **X_train_pca** 에 저장하고, 설명된 분산비율의 합을 **pca_variance_ratio** 에 저장하세요.

In [ ]:
# (32) 여기에 답안코드를 작성하고 실행하세요



### 33. 군집화

DBSCAN 으로 X_train_scaled 를 군집화하여 예측 군집 레이블(이상치는 -1)을 **cluster_labels** 에 저장하세요.

In [ ]:
# (33) 여기에 답안코드를 작성하고 실행하세요



### 34. 딥러닝 설계

다음은 딥러닝 모델을 설계하는 코드입니다. EarlyStopping 에서 '몇 epoch 동안 개선이 없으면 멈출지' 지정하는 파라미터 이름(빈칸 **(A)**)을 파악한 뒤, 아래 코드 전체를 (A)를 채워서 작성하고 실행하세요 (이후 문항들이 model/cb_list 를 사용합니다). (은닉층 활성함수: relu, 출력층: sigmoid/binary_crossentropy, BatchNormalization 포함, ModelCheckpoint 포함)

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cb_list = [EarlyStopping(monitor='val_loss', (A)=14, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))
```

In [ ]:
# (34) 위 코드에서 (A)를 채운 전체 코드를 작성하고 실행하세요



### 35. 딥러닝 학습

위에서 설계한 model 을 batch_size=64, epochs=50 으로 학습하고 history 에 저장하세요 (callbacks=cb_list 사용). 이어서 `model.evaluate(X_valid_scaled, y_valid)` 로 검증 손실/지표를 **eval_loss**, **eval_metric** 에 저장하세요.

In [ ]:
# (35) 여기에 답안코드를 작성하고 실행하세요



### 36. 학습곡선 시각화

history 를 이용해서 학습/검증 **accuracy** 변화를 한 그래프에 시각화하세요 (x축 라벨: epoch, 범례 위치: upper left, 범례 텍스트: train/val).

In [ ]:
# (36) 여기에 답안코드를 작성하고 실행하세요



### 37. 저장된 모델 재사용

ModelCheckpoint 로 저장된 `best_model.keras` 를 `load_model()` 로 다시 불러와서, 검증데이터에 대한 예측값을 **reload_pred** 에 저장하세요.

In [ ]:
# (37) 여기에 답안코드를 작성하고 실행하세요



---
## 자동 채점

위 문항을 모두 풀고 실행한 뒤, 아래 셀을 실행해서 점수를 확인하세요. (딥러닝 문항은 tensorflow 가 필요하므로 Colab 에서 실행했을 때만 채점됩니다.)

In [ ]:
_CHECKS = [(1, '라이브러리 임포트', "all(k in globals() for k in ['pd','np','plt','sns'])", '임포트 확인됨', 'pd/np/plt/sns 임포트가 필요합니다'), (2, '데이터 로드 (read_csv/read_json)', "'my_data' in globals() and tuple(getattr(my_data,'shape',())) == (606, 10) and 'dim_data' in globals() and tuple(getattr(dim_data,'shape',())) == (5, 2)", 'shape 일치', 'my_data shape (606, 10), dim_data shape (5, 2) 이어야 합니다'), (3, '데이터 구조 확인', "'n_rows' in globals() and 'n_cols' in globals() and int(n_rows) == 606 and int(n_cols) == 10", 'n_rows/n_cols 확인', 'n_rows, n_cols 를 my_data.shape 로 올바르게 저장했는지 확인하세요'), (4, '결측치 확인', "_norm_answer(globals().get('answer_4')) == '19'", '정답 일치', '정답은 19 입니다'), (5, '데이터 병합 (pd.merge)', "'data_merged' in globals() and tuple(getattr(data_merged,'shape',())) == (606, 11)", 'shape 일치', 'data_merged shape (606, 11) 이어야 합니다'), (6, '데이터 집계 (groupby)', "'df_grp' in globals() and sorted(str(x) for x in getattr(df_grp,'index',[])) == ['Cat1', 'Cat2', 'Cat3']", '그룹 일치', 'df_grp 의 그룹(index)이 올바르지 않습니다'), (7, '조건부 인덱싱 (loc/iloc)', "'top_quartile_df' in globals() and len(top_quartile_df) == 147 and 'top5_df' in globals() and len(top5_df) == 5", '행 개수 일치', 'top_quartile_df/top5_df 의 행 개수를 확인하세요'), (8, '정렬 (sort_values)', "'sorted_df' in globals() and sorted_df['daily_visitor_count'].dropna().is_monotonic_decreasing", '내림차순 정렬 확인', 'sorted_df 가 daily_visitor_count 기준 내림차순인지 확인하세요'), (9, '컬럼명 변경 (rename)', "'renamed_df' in globals() and 'region_weight' in renamed_df.columns and 'dim_value' not in renamed_df.columns", '컬럼명 변경 확인', 'renamed_df 에 region_weight 컬럼이 있는지 확인하세요'), (10, '데이터 합치기 (concat)', "'concat_df' in globals() and len(concat_df) == 606", 'concat_df 길이 확인', 'concat_df 의 행 개수를 확인하세요'), (11, '피벗/교차분석', "'pivot_result' in globals() and tuple(getattr(pivot_result,'shape',())) == (3, 1)", 'pivot_result shape 확인', 'pivot_result 의 shape 를 확인하세요'), (12, '시각화 (subplots)', "_norm_answer(globals().get('answer_12')) == 'subplots'", '정답 일치', '정답은 subplots 입니다'), (15, '이상치 처리', "'data_temp' in globals() and tuple(getattr(data_temp,'shape',())) == (584, 10)", 'shape 일치', 'data_temp shape (584, 10) 이어야 합니다 ((A)=drop, 전체 코드를 실행했는지 확인)'), (17, '결측치 처리', "'data_na' in globals() and int(data_na.isna().sum().sum()) == 0 and tuple(getattr(data_na,'shape',())) == (584, 11)", '결측치 제거 및 shape 일치', '결측치가 남아있거나 shape 이 올바르지 않습니다'), (21, '인코딩', "'data_preset' in globals() and data_preset.select_dtypes(include='object').shape[1] == 0 and tuple(getattr(data_preset,'shape',())) == (584, 12)", '인코딩 및 shape 일치', '문자열(object) 컬럼이 남아있거나 shape 이 올바르지 않습니다'), (22, '데이터 분리', "all(k in globals() for k in ['X_train','X_valid','y_train','y_valid']) and len(X_train) == 408 and len(X_valid) == 176", '분리 크기 일치', 'X_train/X_valid 길이가 408/176 이어야 합니다'), (23, '스케일링', "'X_train_scaled' in globals() and 'X_valid_scaled' in globals() and len(X_train_scaled) == len(X_train) and len(X_valid_scaled) == len(X_valid)", '스케일링 변수 확인', 'X_train_scaled/X_valid_scaled 를 올바르게 생성했는지 확인하세요'), (24, '스케일러 특성', "len(_norm_answer(globals().get('answer_24'))) > 2", '답변 작성됨 (해설과 비교해 직접 확인하세요)', 'answer_24 가 비어있습니다'), (25, '머신러닝 기본 (fit-predict)', "'model' in globals() and hasattr(model,'predict') and 'pred_y' in globals() and len(pred_y) == len(X_valid) and 'model_score' in globals() and isinstance(model_score,(int,float))", 'model/pred_y/model_score 확인', 'model, pred_y, model_score(=model.score(X_valid,y_valid)) 를 올바르게 생성했는지 확인하세요'), (14, '상관관계 히트맵', "'corr_df' in globals() and hasattr(corr_df,'shape') and corr_df.shape[0] == corr_df.shape[1]", 'corr_df 확인', 'corr_df 를 정사각형 상관계수 행렬로 생성했는지 확인하세요'), (31, '혼동행렬 및 분류 리포트', "'cm' in globals() and hasattr(cm,'shape')", 'cm 확인', 'confusion_matrix 결과를 cm 에 저장했는지 확인하세요'), (32, '차원축소 (PCA)', "'X_train_pca' in globals() and X_train_pca.shape[1] == 2 and 'pca_variance_ratio' in globals() and 0 <= pca_variance_ratio <= 1", 'X_train_pca/분산비율 확인', 'X_train_pca(2차원), pca_variance_ratio 를 확인하세요'), (33, '군집화', "'cluster_labels' in globals() and len(cluster_labels) == 408", 'cluster_labels 길이 확인', 'cluster_labels 의 길이가 X_train_scaled 와 같은지 확인하세요'), (26, 'GridSearch 모델링', "'gs_a' in globals() and hasattr(gs_a,'best_estimator_') and 'gs_b' in globals() and hasattr(gs_b,'best_estimator_')", 'gs_a/gs_b 학습 확인', 'gs_a, gs_b 가 GridSearchCV로 학습되었는지 확인하세요'), (27, 'GridSearch 결과 확인', "_norm_answer(globals().get('answer_27')) == '200'", '정답 일치', '정답은 200 입니다'), (28, '변수중요도', "'fi' in globals() and set(['feature','importance']).issubset(set(fi.columns)) and len(fi) == 11", 'fi 구조 확인', 'fi 에 feature/importance 컬럼과 올바른 길이가 필요합니다'), (29, '성능평가', "'a_score' in globals() and 'b_score' in globals() and isinstance(a_score,(int,float)) and isinstance(b_score,(int,float))", 'a_score/b_score 확인', 'a_score, b_score 를 숫자로 계산했는지 확인하세요'), (30, '모델 성능 비교', "len(_norm_answer(globals().get('answer_30'))) > 0", '답변 작성됨 (해설과 비교해 직접 확인하세요)', 'answer_30 가 비어있습니다'), (34, '딥러닝 설계', "_norm_answer(globals().get('answer_34')) == 'patience' and 'model' in globals() and hasattr(model,'fit') and 'cb_list' in globals()", '정답 및 model/cb_list 확인', "answer_34 는 'patience' 여야 하고, model/cb_list 도 함께 생성해야 합니다 (Colab 실행 필요)"), (35, '딥러닝 학습', "'history' in globals() and hasattr(history,'history') and 'eval_loss' in globals() and 'eval_metric' in globals()", 'history/eval_loss/eval_metric 확인 (Colab 실행 필요)', 'history, eval_loss, eval_metric(=model.evaluate 결과) 를 생성했는지 확인하세요 (Colab 실행 필요)'), (16, '중복 데이터 제거', "'data_temp' in globals() and int(data_temp.duplicated().sum()) == 0 and 'dup_count' in globals()", '중복 제거 확인', 'dup_count 를 계산하고 data_temp 의 중복 행을 제거했는지 확인하세요'), (18, '날짜 특성 추출', "'data_na' in globals() and 'day_of_week' in data_na.columns and 'is_weekend' in data_na.columns and 'event_date' not in data_na.columns", '날짜 파생 컬럼 확인', 'day_of_week/is_weekend 컬럼을 만들고 event_date 원본 컬럼을 삭제했는지 확인하세요'), (19, '파생 변수 생성 (비율)', "'data_na' in globals() and 'ratio_feature' in data_na.columns", 'ratio_feature 확인', 'ratio_feature 컬럼을 생성했는지 확인하세요'), (20, '구간화 (Binning)', "'bin_counts' in globals() and len(bin_counts) == 3", 'bin_counts 확인', '3개 구간으로 나눈 bin_counts 를 생성했는지 확인하세요'), (37, '저장된 모델 재사용', "'reload_pred' in globals() and 'X_valid_scaled' in globals() and len(reload_pred) == len(X_valid_scaled)", 'reload_pred 확인 (Colab 실행 필요)', 'reload_pred 를 생성했는지 확인하세요 (Colab 실행 필요)')]

def _norm_answer(x):
    if x is None:
        return ""
    try:
        return str(x).strip().strip('"\'').lower()
    except Exception:
        return ""

_score = 0
_total = 0
_report = []
for _n, _title, _cond, _ok_msg, _fail_msg in _CHECKS:
    _total += 1
    try:
        _result = bool(eval(_cond))
    except Exception:
        _result = False
    if _result:
        _score += 1
        _report.append(f"[PASS] {_n}번 {_title}: {_ok_msg}")
    else:
        _report.append(f"[FAIL] {_n}번 {_title}: {_fail_msg}")

print("\n".join(_report))
print(f"\n총점: {_score} / {_total}  ({_score/_total*100:.0f}점)" if _total else "채점 가능한 문항이 없습니다")


---
## 해설

채점 후 아래에서 정답과 설명을 확인하세요. 문항 번호가 위 문제 번호와 일치합니다.

### 1번 해설 - 라이브러리 임포트 [코드작성]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

> import ... as ... 문법으로 널리 쓰이는 관례적 별칭을 지정합니다.

### 2번 해설 - 데이터 로드 (read_csv/read_json) [코드작성]

In [ ]:
my_data = pd.read_json('data.json')
dim_data = pd.read_csv('dim.csv')
my_data.head(4)

> pd.read_json() 로 파일 형식에 맞는 로더를 사용합니다.

### 3번 해설 - 데이터 구조 확인 [코드작성]

In [ ]:
my_data.info()
n_rows, n_cols = my_data.shape
print(n_rows, n_cols)
my_data.describe()

> info() 는 컬럼별 타입/결측치를, shape 는 (행, 열) 튜플을 반환합니다.

### 4번 해설 - 결측치 확인 [결과값예측]

In [ ]:
19

> Series.isna().sum() 은 True(결측치)의 개수를 셉니다.

### 5번 해설 - 데이터 병합 (pd.merge) [코드작성]

In [ ]:
data_merged = pd.merge(my_data, dim_data, on='zone_code', how='left')
data_merged.head(4)

> pd.merge(left, right, on=키, how='left') 는 왼쪽 테이블 기준으로 오른쪽 테이블을 결합합니다.

### 6번 해설 - 데이터 집계 (groupby) [코드작성]

In [ ]:
df_grp = data_merged.groupby('location_type')['daily_visitor_count'].mean()
df_grp

> groupby(기준컬럼)[대상컬럼].mean() 형태로 그룹별 집계를 구합니다.

### 7번 해설 - 조건부 인덱싱 (loc/iloc) [코드작성]

In [ ]:
threshold = data_merged['daily_visitor_count'].quantile(0.75)
top_quartile_df = data_merged.loc[data_merged['daily_visitor_count'] >= threshold]
top5_df = top_quartile_df.iloc[:5]

> loc는 조건/라벨 기반, iloc는 정수 위치 기반으로 행을 선택합니다.

### 8번 해설 - 정렬 (sort_values) [코드작성]

In [ ]:
sorted_df = data_merged.sort_values('daily_visitor_count', ascending=False).reset_index(drop=True)

> sort_values(컬럼, ascending=False) 는 해당 컬럼 기준 내림차순 정렬입니다.

### 9번 해설 - 컬럼명 변경 (rename) [코드작성]

In [ ]:
renamed_df = data_merged.rename(columns={'dim_value': 'region_weight'})

> rename(columns={기존명: 새이름}) 은 기본적으로 원본을 바꾸지 않고 새 DataFrame을 반환합니다.

### 10번 해설 - 데이터 합치기 (concat) [코드작성]

In [ ]:
half = len(data_merged) // 2
df_part1 = data_merged.iloc[:half]
df_part2 = data_merged.iloc[half:]
concat_df = pd.concat([df_part1, df_part2], axis=0).reset_index(drop=True)

> pd.concat([df1, df2], axis=0) 은 행 방향으로 데이터프레임을 이어붙입니다 (merge와 달리 키 없이 단순 결합).

### 11번 해설 - 피벗/교차분석 [코드작성]

In [ ]:
pivot_result = data_merged.pivot_table(index='location_type', values='daily_visitor_count', aggfunc='mean')

> pivot_table(index=, values=, aggfunc=) 은 groupby+집계를 표 형태로 재구성합니다.

### 12번 해설 - 시각화 (subplots) [빈칸채우기]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=data_merged, x='location_type', ax=axes[0])
sns.histplot(data=data_merged, x='daily_visitor_count', hue='congestion_level', ax=axes[1])
plt.show()

> plt.subplots() 는 nrows/ncols 로 여러 축(Axes)을 한 번에 만듭니다. 정답: subplots

### 13번 해설 - 시각화 2 [코드작성]

In [ ]:
sns.boxplot(data=data_merged, x='congestion_level', y='daily_visitor_count')
plt.show()

> sns.boxplot(x=, y=) / sns.jointplot(x=, y=) 형태로 그립니다.

### 14번 해설 - 상관관계 히트맵 [코드작성]

In [ ]:
corr_df = data_merged.corr(numeric_only=True)
sns.heatmap(corr_df, annot=True, fmt='.2f')
plt.show()

> DataFrame.corr(numeric_only=True) 로 상관계수 행렬을 구하고 sns.heatmap() 으로 시각화합니다.

### 15번 해설 - 이상치 처리 [빈칸채우기]

In [ ]:
q1 = data_merged['daily_visitor_count'].quantile(0.25)
q3 = data_merged['daily_visitor_count'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
data_temp = data_merged.drop(data_merged[(data_merged['daily_visitor_count'] > upper_fence) | (data_merged['daily_visitor_count'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['location_id'])
data_temp = data_temp.reset_index(drop=True)

> DataFrame.drop() 은 행(기본 axis=0) 또는 열(axis=1)을 삭제합니다. 정답: drop

### 16번 해설 - 중복 데이터 제거 [코드작성]

In [ ]:
dup_count = data_temp.duplicated().sum()
print('중복 행 개수:', dup_count)
data_temp = data_temp.drop_duplicates().reset_index(drop=True)

> DataFrame.duplicated() 는 중복 여부를 불리언 Series 로 반환하며 .sum() 으로 개수를 셀 수 있습니다. drop_duplicates() 는 기본적으로 완전히 동일한 행만 제거합니다.

### 17번 해설 - 결측치 처리 [오류정정]

In [ ]:
fill_value = data_temp['daily_visitor_count'].mode()[0]
data_na = data_temp.fillna({'daily_visitor_count': fill_value})
data_na['day_of_week'] = data_na['day_of_week'].fillna(data_na['day_of_week'].mode()[0])

> [loop_control] 루프/조건 제어 착각: 조건문을 잘못 넣어 일부 로직이 건너뛰어지거나 잘못 실행됨

### 18번 해설 - 날짜 특성 추출 [코드작성]

In [ ]:
data_na['event_date'] = pd.to_datetime(data_na['event_date'])
data_na['day_of_week'] = data_na['event_date'].dt.dayofweek
data_na['is_weekend'] = data_na['day_of_week'].isin([5, 6]).astype(int)
data_na = data_na.drop(columns=['event_date'])
data_na[['day_of_week', 'is_weekend']].head()

> pd.to_datetime() 변환 후 .dt 접근자로 dayofweek 등 날짜 파생값을 뽑아낼 수 있습니다. 원본 문자열 날짜 컬럼은 모델 입력으로 쓸 수 없으므로 파생 컬럼을 만든 뒤 삭제합니다.

### 19번 해설 - 파생 변수 생성 (비율) [코드작성]

In [ ]:
data_na['ratio_feature'] = data_na['daily_visitor_count'] / (data_na['day_of_week'].abs() + 1)
data_na[['ratio_feature']].describe()

> 두 수치형 컬럼을 조합해 새로운 파생 변수(비율/상호작용 특성)를 만드는 특성공학 기법입니다.

### 20번 해설 - 구간화 (Binning) [코드작성]

In [ ]:
numeric_col_bin = pd.cut(data_na['daily_visitor_count'], bins=3, labels=['Low', 'Medium', 'High'])
bin_counts = numeric_col_bin.value_counts()
print(bin_counts)

> pd.cut(컬럼, bins=N, labels=[...]) 은 값의 범위를 N개의 동일 너비 구간으로 나눕니다 (qcut은 분위수 기준이라 값이 중복될 때 오류가 날 수 있어, 균등폭 구간에는 cut을 흔히 씁니다).

### 21번 해설 - 인코딩 [코드작성]

In [ ]:
data_preset = pd.get_dummies(data=data_na, columns=['location_type'], drop_first=True)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data_preset['zone_code'] = le.fit_transform(data_preset['zone_code'])
from sklearn.preprocessing import LabelEncoder as _TargetLE
data_preset['congestion_level'] = _TargetLE().fit_transform(data_preset['congestion_level'])
data_preset.info()

> 저-카디널리티는 원-핫, 코드성 범주는 라벨 인코딩을 흔히 사용합니다. 타깃 컬럼도 XGBoost/LightGBM 등 일부 모델은 문자열 클래스를 그대로 받아들이지 않으므로, `congestion_level` 도 함께 LabelEncoder 로 0/1(다중클래스는 0..N-1) 정수로 인코딩합니다.

### 22번 해설 - 데이터 분리 [코드작성]

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preset.drop(['congestion_level'], axis=1)
y = data_preset['congestion_level']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=7, stratify=y)

> train_test_split(X, y, ...) 은 X_train, X_valid, y_train, y_valid 순서로 반환합니다.

### 23번 해설 - 스케일링 [오류정정]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

> [scope_reference] 이 코드는 스케일러를 학습(fit)하는 순서가 잘못되었습니다. 표준적인 데이터 분할에서는 스케일러는 훈련 데이터(X_train)만을 사용하여 학습되어야 하며, 그 후 훈련 및 검증 데이터에 변환을 적용해야 합니다. 이 버그는 검증 데이터에 대해 훈련 데이터의 스케일링 파라미터를 잘못 적용하게 만듭니다.

### 24번 해설 - 스케일러 특성 [결과값예측]

In [ ]:
'평균(mean) 0, 표준편차(std) 1에 가까워집니다 (Z-score 표준화).'

> StandardScaler 의 정의에 따른 이론적 특성입니다.

### 25번 해설 - 머신러닝 기본 (fit-predict) [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=7)
model.fit(X_train_scaled, y_train)
pred_y = model.predict(X_valid_scaled)
model_score = model.score(X_valid_scaled, y_valid)
print(pred_y[:5], model_score)

> import → model = 클래스() → model.fit(X_train, y_train) → pred_y = model.predict(X_valid) → model.score(X_valid, y_valid) 5줄 템플릿입니다. score() 는 분류=accuracy, 회귀=R² 를 반환합니다.

### 26번 해설 - GridSearch 모델링 [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import RandomizedSearchCV

gs_a = RandomizedSearchCV(DecisionTreeClassifier(random_state=7), {'max_depth':[3,5,7]}, cv=10, scoring='f1_macro', n_iter=5, random_state=7)
gs_a.fit(X_train_scaled, y_train)

gs_b = RandomizedSearchCV(ExtraTreesClassifier(random_state=7), {'n_estimators':[50, 100, 200], 'max_depth':[3,5,7]}, cv=10, scoring='f1_macro', n_iter=5, random_state=7)
gs_b.fit(X_train_scaled, y_train)

> RandomizedSearchCV(estimator, param_grid/distributions, cv=...).fit(X_train, y_train) 형태로 탐색합니다. ExtraTreesClassifier 는 sklearn 기본 앙상블 외에 XGBoost/LightGBM 계열일 수도 있습니다. RandomizedSearchCV는 격자 전체가 아닌 n_iter 만큼만 무작위로 조합을 시도해 더 빠릅니다.

### 27번 해설 - GridSearch 결과 확인 [결과값예측]

In [ ]:
200

> 제시된 후보 중 GridSearch가 고를 수 있는 최댓값을 묻는 문항입니다.

### 28번 해설 - 변수중요도 [오류정정]

In [ ]:
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=True)[:15]
sns.barplot(x='feature', y='importance', data=fi)
plt.show()

> [operator_precedence] 기존 코드는 모델의 중요도 점수를 그대로 사용했지만, 오류가 삽입된 코드에서는 `feature_importances_`에 100으로 나누는 연산을 추가했습니다. 이로 인해 실제 특성 중요도가 1/100로 왜곡되어 시각화 결과가 부정확해집니다.

### 29번 해설 - 성능평가 [코드작성]

In [ ]:
from sklearn.metrics import roc_auc_score

proba_a = gs_a.best_estimator_.predict_proba(X_valid_scaled)[:, 1]
proba_b = gs_b.best_estimator_.predict_proba(X_valid_scaled)[:, 1]

a_score = roc_auc_score(y_valid, proba_a)
b_score = roc_auc_score(y_valid, proba_b)
print(a_score, b_score)

> roc_auc_score(정답, 예측확률) 형태이며 predict()가 아닌 predict_proba() 의 확률값을 사용합니다. 타깃이 이미 0/1 정수이므로 별도 이진화가 필요 없습니다.

### 30번 해설 - 모델 성능 비교 [결과값예측]

In [ ]:
'실행 결과에 따라 달라집니다: 값이 더 큰 쪽 이 더 우수한 모델입니다. a_score, b_score 를 직접 비교해서 판단하세요.'

> 정확도/F1/ROC-AUC 등은 높을수록, MAE/MSE 등 오차 지표는 낮을수록 좋은 성능입니다.

### 31번 해설 - 혼동행렬 및 분류 리포트 [코드작성]

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_b = gs_b.best_estimator_.predict(X_valid_scaled)
cm = confusion_matrix(y_valid, y_pred_b)
sns.heatmap(cm, annot=True, fmt='d')
plt.show()
print(classification_report(y_valid, y_pred_b))

> confusion_matrix(실제값, 예측값) 은 클래스별 정오답 개수를, classification_report 는 precision/recall/f1 을 클래스별로 요약해 보여줍니다.

### 32번 해설 - 차원축소 (PCA) [코드작성]

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
pca_variance_ratio = pca.explained_variance_ratio_.sum()
print(X_train_pca.shape, pca_variance_ratio)

> PCA(n_components=k) 는 고차원 데이터를 k개의 주성분으로 축소하며, explained_variance_ratio_ 는 각 주성분이 설명하는 분산 비율입니다.

### 33번 해설 - 군집화 [코드작성]

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=1.5, min_samples=5)
cluster_labels = db.fit_predict(X_train_scaled)
print(pd.Series(cluster_labels).value_counts())

> DBSCAN 은 밀도 기반 군집분석으로, 군집 개수를 미리 정하지 않아도 되고 이상치를 -1로 표시합니다.

### 34번 해설 - 딥러닝 설계 [빈칸채우기]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = Sequential([
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cb_list = [EarlyStopping(monitor='val_loss', patience=14, restore_best_weights=True)]
cb_list.append(ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True))

> EarlyStopping(patience=N) 은 N번 연속 개선이 없으면 학습을 멈춥니다. 정답: patience

### 35번 해설 - 딥러닝 학습 [코드작성]

In [ ]:
history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=64,
                    validation_data=(X_valid_scaled, y_valid), callbacks=cb_list)

eval_loss, eval_metric = model.evaluate(X_valid_scaled, y_valid)
print('검증 loss:', eval_loss, '/ 검증 metric:', eval_metric)

> model.fit(X, y, epochs=, batch_size=, validation_data=, callbacks=) 형태입니다. model.evaluate(X, y) 는 (loss, metric) 튜플을 반환합니다.

### 36번 해설 - 학습곡선 시각화 [코드작성]

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

> history.history[지표] 로 epoch별 기록을 꺼내 plt.plot() + plt.legend(loc=...) 으로 그립니다.

### 37번 해설 - 저장된 모델 재사용 [코드작성]

In [ ]:
from tensorflow.keras.models import load_model

saved_model = load_model('best_model.keras')
reload_pred = saved_model.predict(X_valid_scaled)
reload_pred[:5]

> 학습 없이도 저장된 가중치를 불러와(load_model) 바로 predict() 할 수 있습니다.